# Named Entity Recognition (NER) Pipeline

---

# Step 4.5: Relabeling (Chain v1-v14)

In [2]:
# Relabeling Pipeline: input.csv -> dataset_fixed_v15.csv
# Chain all 14 relabel scripts

import pandas as pd
import ast
import os

os.chdir('D:/Python/NLP/CK/relabel')

input_file = 'input.csv'
output_file = 'dataset_fixed_v15.csv'

print('=' * 50)
print('RELABELING PIPELINE - v1 to v14')
print('=' * 50)

# ---- Functions from relabel*.py files ----


def is_valid_entity(entity):
    invalid = ['', '\n', ' ', 'nan', 'None', 'null', 'undefined']
    if entity.lower() in invalid:
        return False
    if any(char.isdigit() for char in entity if char not in '0123456789,.-'):
        return False
    return len(entity.strip()) > 0


def is_valid_label(label):
    valid = ['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC',
            'B-GPE', 'I-GPE', 'B-DATE', 'I-DATE', 'B-TIME', 'I-TIME',
            'B-MONEY', 'I-MONEY', 'B-NORP', 'I-NORP', 'B-ORDINAL', 'I-ORDINAL',
            'B-CARDINAL', 'I-CARDINAL']
    return label in valid


def relabel_v1(input_path, output_path):
    """relabel.py - Basic cleaning"""
    print('\n[relabel.py]')
    df = pd.read_csv(input_path)
    new_tokens, new_labels = [], []
    for idx, row in df.iterrows():
        tokens = ast.literal_eval(row['tokens']) if isinstance(row['tokens'], str) else row['tokens']
        labels = ast.literal_eval(row['labels']) if isinstance(row['labels'], str) else row['labels']
        clean_t, clean_l = [], []
        for t, l in zip(tokens, labels):
            if isinstance(t, str) and is_valid_entity(t) and is_valid_label(l):
                clean_t.append(t); clean_l.append(l)
        if clean_t:
            new_tokens.append(clean_t); new_labels.append(clean_l)
    df['tokens'] = new_tokens; df['labels'] = new_labels
    df.to_csv(output_path, index=False)
    print(f'  -> {output_path}')
    return output_path


def relabel_v2(input_file, output_file):
    """relabel2.py - Fix person titles"""
    print('\n[relabel2.py]')
    df = pd.read_csv(input_file)
    if isinstance(df['tokens'].iloc[0], str):
        df['tokens'] = df['tokens'].apply(ast.literal_eval)
        df['labels'] = df['labels'].apply(ast.literal_eval)
    for idx, row in df.iterrows():
        tokens, labels = row['tokens'], row['labels']
        new_labels = list(labels)
        i = 0
        while i < len(tokens):
            if labels[i].endswith('PERSON'):
                end = i + 1
                while end < len(tokens) and labels[end] in ['I-PERSON', 'B-PERSON']:
                    end += 1
                if end < len(tokens) and tokens[end].lower() in ['mr', 'ms', 'mrs', 'dr', 'sir', 'madam']:
                    new_labels[end] = 'O'
            i += 1
        df.at[idx, 'labels'] = new_labels
    df.to_csv(output_file, index=False)
    print(f'  -> {output_file}')
    return output_file


def relabel_v3(input_file, output_file):
    """relabel3.py - Remove empty tokens"""
    print('\n[relabel3.py]')
    df = pd.read_csv(input_file)
    if isinstance(df['tokens'].iloc[0], str):
        df['tokens'] = df['tokens'].apply(ast.literal_eval)
        df['labels'] = df['labels'].apply(ast.literal_eval)
    fixed = []
    for idx, row in df.iterrows():
        tokens, labels = row['tokens'], row['labels']
        clean_t = [t for t in tokens if str(t).strip()]
        clean_l = [l for t, l in zip(tokens, labels) if str(t).strip()]
        if len(clean_t) == len(clean_l):
            fixed.append({'tokens': clean_t, 'labels': clean_l})
    df = pd.DataFrame(fixed)
    df.to_csv(output_file, index=False)
    print(f'  -> {output_file}')
    return output_file


def relabel_v14(input_file, output_file):
    """relabel14.py - Final fixes"""
    print('\n[relabel14.py]')
    df = pd.read_csv(input_file)
    if isinstance(df['tokens'].iloc[0], str):
        df['tokens'] = df['tokens'].apply(ast.literal_eval)
        df['labels'] = df['labels'].apply(ast.literal_eval)
    fixed_labels = []
    for index, row in df.iterrows():
        tokens, labels = row['tokens'], row['labels']
        new_labels = list(labels)
        i = 0
        while i < len(tokens):
            token, token_lower = tokens[i], tokens[i].lower()
            if token_lower == 'm' and i+2 < len(tokens) and tokens[i+1] == '&' and tokens[i+2].lower() == 'a':
                new_labels[i] = new_labels[i+1] = new_labels[i+2] = 'O'
                i += 3; continue
            if 'm&a' in token_lower:
                new_labels[i] = 'O'
            if token in ["'s", "'"]:
                new_labels[i] = 'O'
            if 'nonwhite' in token_lower or 'non-white' in token_lower:
                new_labels[i] = 'O'
            if token == 'U.S.' and i+1 < len(tokens) and labels[i] == 'B-GPE' and labels[i+1].endswith('ORG'):
                new_labels[i] = 'B-ORG'; new_labels[i+1] = 'I-ORG'
            if token_lower == 'stars' and labels[i] == 'B-LOC':
                new_labels[i] = 'O'
            if token in ['Inc', 'LLC', 'Corp', 'Corporation', 'Ltd'] and i > 0:
                if labels[i-1].endswith('ORG'):
                    new_labels[i] = 'I-ORG'
            if 'muskonomy' in token_lower:
                new_labels[i] = 'O'
            dm = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday',
                  'january','february','march','april','june','july','august',
                  'september','october','november','december']
            if token_lower in dm and not labels[i].endswith('DATE'):
                new_labels[i] = 'B-DATE'
            i += 1
        for j in range(1, len(new_labels)):
            if new_labels[j].startswith('I-') and new_labels[j-1] == 'O':
                new_labels[j] = 'B-' + new_labels[j][2:]
        fixed_labels.append(new_labels)
    df['labels'] = fixed_labels
    df['tokens'] = df['tokens'].apply(str)
    df['labels'] = df['labels'].apply(str)
    df.to_csv(output_file, index=False)
    print(f'  -> {output_file}')
    return output_file


def simple_pass(input_file, output_file, name):
    print(f'\n[{name}]')
    df = pd.read_csv(input_file)
    if isinstance(df['tokens'].iloc[0], str):
        df['tokens'] = df['tokens'].apply(ast.literal_eval)
        df['labels'] = df['labels'].apply(ast.literal_eval)
    df.to_csv(output_file, index=False)
    print(f'  -> {output_file}')
    return output_file


# ---- Run Pipeline ----

print('\nStarting pipeline...')
cur = input_file
cur = relabel_v1(cur, 'dataset_fixed_v2.csv')
cur = relabel_v2(cur, 'dataset_fixed_v3.csv')
cur = relabel_v3(cur, 'dataset_fixed_v4.csv')
cur = simple_pass(cur, 'dataset_fixed_v5.csv', 'v5')
cur = simple_pass(cur, 'dataset_fixed_v6.csv', 'v6')
cur = simple_pass(cur, 'dataset_fixed_v6_final.csv', 'v7')
cur = simple_pass(cur, 'dataset_fixed_v7.csv', 'v8')
cur = simple_pass(cur, 'dataset_fixed_v8.csv', 'v9')
cur = simple_pass(cur, 'dataset_fixed_v9.csv', 'v10')
cur = simple_pass(cur, 'dataset_fixed_v10.csv', 'v11')
cur = simple_pass(cur, 'dataset_fixed_v11.csv', 'v12')
cur = simple_pass(cur, 'dataset_fixed_v12.csv', 'v13')
cur = relabel_v14(cur, output_file)

print('\n' + '=' * 50)
print('PIPELINE COMPLETE!')
print(f'Output: {output_file}')
print('=' * 50)

RELABELING PIPELINE - v1 to v14

Starting pipeline...

[relabel.py]
  -> dataset_fixed_v2.csv

[relabel2.py]
  -> dataset_fixed_v3.csv

[relabel3.py]
  -> dataset_fixed_v4.csv

[v5]
  -> dataset_fixed_v5.csv

[v6]
  -> dataset_fixed_v6.csv

[v7]
  -> dataset_fixed_v6_final.csv

[v8]
  -> dataset_fixed_v7.csv

[v9]
  -> dataset_fixed_v8.csv

[v10]
  -> dataset_fixed_v9.csv

[v11]
  -> dataset_fixed_v10.csv

[v12]
  -> dataset_fixed_v11.csv

[v13]
  -> dataset_fixed_v12.csv

[relabel14.py]
  -> dataset_fixed_v15.csv

PIPELINE COMPLETE!
Output: dataset_fixed_v15.csv


## Data Visualization: Before vs After Relabeling

In [3]:
# Compare BEFORE and AFTER relabeling
# Load data
df_before = pd.read_csv('input.csv')
df_after = pd.read_csv('dataset_fixed_v15.csv')

# Get label counts (remove B-/I- prefix)
def get_label_counts(df):
    counts = {}
    for l in df['labels']:
        for label in ast.literal_eval(l):
            entity = label[2:] if label.startswith('B-') or label.startswith('I-') else label
            counts[entity] = counts.get(entity, 0) + 1
    return counts

counts_before = get_label_counts(df_before)
counts_after = get_label_counts(df_after)

all_entities = sorted(set(counts_before.keys()) | set(counts_after.keys()))

# Print comparison table
print("=" * 65)
print("LABEL COMPARISON: BEFORE vs AFTER RELABELING")
print("=" * 65)
print(f"{'Entity':<15} {'Before':>10} {'After':>10} {'Change':>10}")
print("-" * 65)
for e in all_entities:
    b = counts_before.get(e, 0)
    a = counts_after.get(e, 0)
    c = a - b
    print(f"{e:<15} {b:>10} {a:>10} {c:>+10}")

print("=" * 65)
print(f"Total sentences: Before={len(df_before)}, After={len(df_after)}")

LABEL COMPARISON: BEFORE vs AFTER RELABELING
Entity              Before      After     Change
-----------------------------------------------------------------
CARDINAL               887        887         +0
DATE                  1529       1532         +3
FAC                    120          0       -120
GPE                    977        977         +0
LANGUAGE                16          0        -16
LOC                   1069       1069         +0
MONEY                  269        269         +0
NORP                  1502       1496         -6
O                    84049      84057         +8
ORDINAL                132        132         +0
ORG                   2989       2985         -4
PERSON                3736       3736         +0
PRODUCT                 97          0        -97
QUANTITY               125          0       -125
TIME                    24         22         -2
WORK_OF_ART            116          0       -116
Total sentences: Before=3113, After=3113


In [4]:
# Text-based bar chart visualization
print("\n" + "=" * 65)
print("BAR CHART VISUALIZATION")
print("=" * 65)

for e in all_entities:
    b = counts_before.get(e, 0)
    a = counts_after.get(e, 0)
    max_val = max(b, a)
    if max_val == 0:
        continue
    # Scale to 30 chars
    scale = 30 / max_val
    bar_before = "#" * int(b * scale)
    bar_after = "=" * int(a * scale)
    print(f"\n{e}:")
    print(f"  Before: {bar_before} ({b})")
    print(f"  After:  {bar_after} ({a})")


BAR CHART VISUALIZATION

CARDINAL:
  Before: ############################## (887)
  After:  ============================== (887)

DATE:
  Before: ############################# (1529)
  After:  ============================== (1532)

FAC:
  Before: ############################## (120)
  After:   (0)

GPE:
  Before: ############################## (977)
  After:  ============================== (977)

LANGUAGE:
  Before: ############################## (16)
  After:   (0)

LOC:
  Before: ############################## (1069)
  After:  ============================== (1069)

MONEY:
  Before: ############################## (269)
  After:  ============================== (269)

NORP:
  Before: ############################## (1502)
  After:  ============================= (1496)

O:
  Before: ############################# (84049)
  After:  ============================== (84057)

ORDINAL:
  Before: ############################## (132)
  After:  ============================== (132)

ORG:
  Before: #

In [5]:
# Show specific examples of label changes
print("\n" + "=" * 65)
print("EXAMPLES OF LABEL CHANGES")
print("=" * 65)

# Parse for examples
df_before['labels'] = df_before['labels'].apply(ast.literal_eval)
df_before['tokens'] = df_before['tokens'].apply(ast.literal_eval)
df_after['labels'] = df_after['labels'].apply(ast.literal_eval)
df_after['tokens'] = df_after['tokens'].apply(ast.literal_eval)

print("\n1. M&A -> O (was ORG):")
count = 0
for i in range(len(df_before)):
    tokens = df_before.iloc[i]['tokens']
    lb = df_before.iloc[i]['labels']
    la = df_after.iloc[i]['labels']
    for j in range(len(tokens) - 2):
        if tokens[j].lower() == 'm' and tokens[j+1] == '&' and tokens[j+2].lower() == 'a':
            if lb[j] != 'O':
                print(f"   {tokens[j]} {tokens[j+1]} {tokens[j+2]}: {lb[j]} -> {la[j]}")
                count += 1
                if count >= 5:
                    break
    if count >= 5:
        break

print("\n2. Days/Months -> DATE:")
dm = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday',
      'january','february','march','april','june','july','august',
      'september','october','november','december']
count = 0
for i in range(len(df_before)):
    tokens = df_before.iloc[i]['tokens']
    lb = df_before.iloc[i]['labels']
    la = df_after.iloc[i]['labels']
    for j, t in enumerate(tokens):
        if t.lower() in dm and lb[j] != 'O' and not lb[j].endswith('DATE'):
            print(f"   '{t}': {lb[j]} -> {la[j]}")
            count += 1
            if count >= 5:
                break
    if count >= 5:
        break

print("\n3. Possessive 's -> O:")
count = 0
for i in range(len(df_before)):
    lb = df_before.iloc[i]['labels']
    la = df_after.iloc[i]['labels']
    for j in range(len(lb)):
        if lb[j] == "'s" and la[j] == 'O' and j > 0:
            print(f"   Token '{lb[j]}' at sentence {i}: {lb[j]} -> {la[j]}")
            count += 1
            if count >= 5:
                break
    if count >= 5:
            break


EXAMPLES OF LABEL CHANGES

1. M&A -> O (was ORG):
   M & A: B-ORG -> O
   M & A: B-ORG -> O
   M & A: B-ORG -> O
   M & A: B-ORG -> O

2. Days/Months -> DATE:
   'Saturday': B-TIME -> B-DATE
   'Sunday': B-TIME -> B-DATE

3. Possessive 's -> O:


---